# YOLOv8n Resolution Experiment, Dataset V1
> Notebook note: This notebook documents the completed 960-pixel training and input-resolution comparison. The original model weights are preserved externally. Retraining is disabled by default.
## Purpose
Investigate whether higher image resolution improves detection on cleaned Roboflow Dataset V1. Two different operations must not be confused:
1. Train at 960 px: train a new YOLOv8n model at imgsz=960 with mosaic disabled, producing yolov8n_v1_no_mosaic_960/weights/best.pt.
2. Evaluate at 960 px: use an existing model trained at 640 px and change only the validation input resolution to imgsz=960.

These are separate experiments, although they use the same cleaned V1 validation set (119 images, 154 labelled boxes). The training/evaluation data and the saved checkpoints are not included in GitHub; see the main repository README for the dataset source and shared model weights.
## File paths
- 960-trained checkpoint: model_runs/yolov8n_v1_no_mosaic_960/weights/best.pt
- 640-trained no-mosaic checkpoint: model_runs/yolov8n_v1_no_mosaic/weights/best.pt
- 640-trained-at-960 validation output: model_runs/v1_no_mosaic_640_at960/

The paths below refer to the original Colab Drive structure. Update them if running from another account.

In [1]:
# Dataset access
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

# Dataset and model files on Google Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/MMA3001_Project")
ZIP_PATH = PROJECT_ROOT / "dataset" / "pork_rasher_v1.zip"
CLEAN_DIR = Path("/content/pork_rasher_dataset_v1_clean")
MANIFEST_PATH = PROJECT_ROOT / "dataset" / "manifests" / "v1_preparation.json"
RUNS_DIR = PROJECT_ROOT / "model_runs"

# Shared dataset preparation code.
module_folders = [PROJECT_ROOT / "src", Path.cwd() / "src", Path("/content/src"), Path("/content")]
module_dir = next((p for p in module_folders if (p / "dataset_preparation.py").is_file()), None)
if module_dir is None:
    raise FileNotFoundError("dataset_preparation.py not found in the project src folder")
sys.path.insert(0, str(module_dir))
from dataset_preparation import prepare_dataset, print_audit, assert_preparations_match

Mounted at /content/drive


In [2]:
# Remove overlapping source images from Dataset V1.
manifest = prepare_dataset(
    zip_path=ZIP_PATH,
    output_dir=CLEAN_DIR,
    expected_dataset_version=1,
    require_labels=True,
)
print("Removed overlaps:", manifest["removed_overlap_images"])
print_audit(manifest["audit"])

# Verify the cleaned split sizes and class names.
expected_images = {"train": 1935, "valid": 119, "test": 80}
expected_classes = ["loose-meat", "packaging-error", "twisted-meat", "unsealed"]
image_counts = {s: x["images"] for s, x in manifest["audit"]["splits"].items()}
assert image_counts == expected_images, (image_counts, expected_images)
assert manifest["audit"]["class_names"] == expected_classes
assert manifest["moved_to_validation"] == 0
clean = CLEAN_DIR
print("Dataset V1 checked.")

Removed overlaps: {'train': 9, 'valid': 1}

TRAIN: 1935 images; 29 backgrounds; 0 missing label files
  loose-meat: 139 boxes
  packaging-error: 931 boxes
  twisted-meat: 91 boxes
  unsealed: 1107 boxes

VALID: 119 images; 0 backgrounds; 0 missing label files
  loose-meat: 5 boxes
  packaging-error: 57 boxes
  twisted-meat: 5 boxes
  unsealed: 87 boxes

TEST: 80 images; 1 backgrounds; 0 missing label files
  loose-meat: 3 boxes
  packaging-error: 37 boxes
  twisted-meat: 7 boxes
  unsealed: 52 boxes
Dataset V1 checked.


In [3]:
# Verify the dataset preparation record, if available.
if MANIFEST_PATH.exists():
    assert_preparations_match(manifest, MANIFEST_PATH)
    print("Preparation record matches:", MANIFEST_PATH)
else:
    print("No existing preparation record; split counts and classes were checked.")

Preparation record matches: /content/drive/MyDrive/MMA3001_Project/dataset/manifests/v1_preparation.json


## 960-Pixel Training
The 960-pixel model is configured for 30 epochs with mosaic augmentation disabled.

In [4]:
%pip install -q ultralytics
from ultralytics import YOLO

# YOLOv8n training at 960-pixel resolution.
RETRAIN_960 = False
if RETRAIN_960:
    model_960 = YOLO("yolov8n.pt")
    results_960 = model_960.train(
        data=str(clean / "data.yaml"),
        epochs=30,
        imgsz=960,
        batch=16,
        device=0,
        seed=42,
        mosaic=0.0,
        project=str(RUNS_DIR),
        name="yolov8n_v1_no_mosaic_960",
    )
    print("Training output:", results_960.save_dir)
else:
    print("960-pixel retraining skipped; saved weights are unchanged.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 7.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
960-pixel retraining skipped; saved weights are unchanged.


## Recorded Results: Moel Trained at 960 px
The completed 30-epoch training log (original V1 checking notebook) reports the following best-checkpoint validation metrics on 119 V1 validation images (154 defect annotations):
| Metric | Result |
|---|---:|
| Precision | 58.3% |
| Recall | 55.3% |
| mAP50 | 57.7% |
| mAP50–95 | 32.1% |
| Packaging-error recall | 38.6% |

Training configuration: YOLOv8n, 30 epochs, imgsz=960, batch=16, seed=42, mosaic=0.0. The completed run saved its selected checkpoint as yolov8n_v1_no_mosaic_960/weights/best.pt.

These are validation results, not independent test-set results. The record is transcribed from the saved output of the original completed training cell; the 960 training does not have to be repeated to inspect the model.

## Optional Check: 960-Trained Checkpoint
Turn on RECHECK_960_TRAINED to independently evaluate the existing 960-trained checkpoint. The optional step saves fresh validation metrics and plots; it does not retrain the model. Ensure the original weights and cleaned V1 dataset are accessible first.

In [5]:
# Optional: verify the saved 960-trained checkpoint on the V1 validation split.
# This loads the existing trained weights; it does not train a model.
RECHECK_960_TRAINED = False
if RECHECK_960_TRAINED:
    checkpoint_960 = RUNS_DIR / "yolov8n_v1_no_mosaic_960" / "weights" / "best.pt"
    if not checkpoint_960.is_file():
        raise FileNotFoundError(f"960-trained model not found: {checkpoint_960}")
    model_960_saved = YOLO(str(checkpoint_960))
    metrics_960 = model_960_saved.val(
        data=str(clean / "data.yaml"),
        imgsz=960,
        split="val",
        conf=0.001,
        iou=0.7,
        plots=True,
        project=str(RUNS_DIR),
        name="v1_no_mosaic_960_recheck",
        exist_ok=True,
    )
    print("Validation mAP50:", metrics_960.box.map50)
else:
    print("Optional validation of the saved 960-trained model skipped.")

Optional validation of the saved 960-trained model skipped.


## Evaluate the 640-Trained Model at 960 Pixels

In [6]:
# Optional: validate the 640-trained checkpoint using 960-pixel inputs.
# This is not equivalent to a newly trained 960-pixel model.
RECHECK_640_AT960 = False
if RECHECK_640_AT960:
    model_path = RUNS_DIR / "yolov8n_v1_no_mosaic" / "weights" / "best.pt"
    if not model_path.is_file():
        raise FileNotFoundError(f"640-trained no-mosaic weights not found: {model_path}")
    model_640 = YOLO(str(model_path))
    results_640_at960 = model_640.val(
        data=str(clean / "data.yaml"),
        imgsz=960,
        split="val",
        conf=0.001,
        iou=0.7,
        plots=True,
        project=str(RUNS_DIR),
        name="v1_no_mosaic_640_at960",
        exist_ok=True,
    )
    print("Validation mAP50:", results_640_at960.box.map50)
else:
    print("Optional 640-trained-at-960 validation skipped.")

Optional 640-trained-at-960 validation skipped.


## Validation Comparison and Interpretation
| Experiment | Validation precision | Recall | mAP50 | mAP50–95 | Packaging-error recall |
|---|---:|---:|---:|---:|---:|
| **960-trained no-mosaic model, evaluated at 960** | 58.3% | 55.3% | 57.7% | 32.1% | 38.6% |
| **640-trained no-mosaic model, evaluated at 960** | 61.7% | 55.3% | 57.7% | 23.0% | 21.1% |

The 640-trained model evaluated at its native 640-pixel resolution separately recorded validation recall 62.3% and mAP50 69.5% in the project's V1 evaluation results. Thus, using a larger input did not automatically improve overall detection performance. The 960-trained model improved packaging-error recall compared with evaluating the 640-trained model at 960 inputs (38.6% versus 21.1%), but this observation is specific to the recorded validation conditions and class counts.

Limitations: These results are from the validation split, not an independent 960-model test evaluation. Image resolution, training resolution and confidence settings must be stated when comparing results. The 960 model was not chosen for the final inference demonstration; the selected demonstration remains V1 no-mosaic 640 px.